## Import Libraries

In [59]:
import ast
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer ,OneHotEncoder,StandardScaler,MultiLabelBinarizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import make_scorer, f1_score

## Call Clean Data

In [60]:
df_balanced=pd.read_csv(r"..\data\processed\Data_Entry_2017_Cleaned.csv")
df_balanced["finding_labels"] = df_balanced["finding_labels"].apply(ast.literal_eval)


In [61]:
df_balanced

,image_id,finding_labels,follow_up,patient_id,patient_age,patient_gender,view_position,image_width,image_height,pixel_spacing_x,pixel_spacing_y
0,00003823_000.png,[No Finding],0,3823,66,F,PA,2048,2500,0.171,0.171
1,00004381_035.png,[Infiltration],35,4381,25,M,PA,2992,2991,0.143,0.143
2,00011884_001.png,[No Finding],1,11884,24,M,AP,3056,2544,0.139,0.139
3,00024887_000.png,[No Finding],0,24887,36,M,PA,2992,2991,0.143,0.143
4,00013208_000.png,[No Finding],0,13208,60,M,PA,2992,2991,0.143,0.143
...,...,...,...,...,...,...,...,...,...,...,...
52570,00007008_001.png,[No Finding],1,7008,29,M,AP,2500,2048,0.171,0.171
52571,00025498_001.png,[Infiltration],1,25498,86,M,AP,3056,2544,0.139,0.139
52572,00000042_004.png,[No Finding],4,42,71,M,AP,2808,2544,0.139,0.139
52573,00012716_000.png,[No Finding],0,12716,40,M,PA,2986,2991,0.143,0.143


In [62]:
df_balanced.info()

<class 'pandas.DataFrame'>
RangeIndex: 52575 entries, 0 to 52574
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   image_id         52575 non-null  str    
 1   finding_labels   52575 non-null  object 
 2   follow_up        52575 non-null  int64  
 3   patient_id       52575 non-null  int64  
 4   patient_age      52575 non-null  int64  
 5   patient_gender   52575 non-null  str    
 6   view_position    52575 non-null  str    
 7   image_width      52575 non-null  int64  
 8   image_height     52575 non-null  int64  
 9   pixel_spacing_x  52575 non-null  float64
 10  pixel_spacing_y  52575 non-null  float64
dtypes: float64(2), int64(5), object(1), str(3)
memory usage: 4.4+ MB


## Data Splitting

In [63]:
X = df_balanced.drop(["finding_labels", "image_id", "patient_id"],axis=1)
y=df_balanced["finding_labels"]

In [64]:
X_train,X_test,y_train,y_test=train_test_split(X,y,train_size=0.7,random_state=42)

In [65]:
len(X_train),len(X_test)

(36802, 15773)

## Preprocessing

In [66]:
categorical_cols=["view_position","patient_gender"]
numerical_cols=["patient_age","image_width","image_height","pixel_spacing_x","pixel_spacing_y"]
log_pipeline=Pipeline([("log", FunctionTransformer(np.log1p)),
                       ("scaler",StandardScaler())])

In [67]:
preprocessor = ColumnTransformer([
    ("log", log_pipeline, ["follow_up"]),
    ("num", StandardScaler(), numerical_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])

In [68]:
classes = ["Atelectasis", "Effusion", "Infiltration"]
mlb = MultiLabelBinarizer(classes=classes)
y_train = mlb.fit_transform(y_train)
y_test = mlb.transform(y_test)

d:\depi_graduation_project\MediVision-AI\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:1016: UserWarning: unknown class(es) ['No Finding'] will be ignored
  warnings.warn(


## Ml model

In [69]:
models = {
    "Logistic Regression": OneVsRestClassifier(
        LogisticRegression(max_iter=1000, class_weight="balanced")
    ),

    "Random Forest": OneVsRestClassifier(
        RandomForestClassifier(
            n_estimators=200,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )
    ),

    "XGBoost": OneVsRestClassifier(
        XGBClassifier(
            n_estimators=200,
            max_depth=5,
            learning_rate=0.05,
            scale_pos_weight=4,
            random_state=42,
            eval_metric="logloss",
            n_jobs=-1
        )
    )
}

In [70]:
scoring = make_scorer(
    f1_score,
    average="macro",
    zero_division=0
)

results = {}

groups = df_balanced.loc[X_train.index, "patient_id"]

for name, classifier in models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", classifier)
    ])

    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=5,
        groups=groups,
        scoring=scoring,
        n_jobs=-1
    )

    results[name] = {
        "Mean F1 Macro": scores.mean(),
        "Std F1 Macro": scores.std()
    }

results

d:\depi_graduation_project\MediVision-AI\.venv\Lib\site-packages\sklearn\model_selection\_split.py:86: UserWarning: The groups parameter is ignored by KFold
  warnings.warn(
d:\depi_graduation_project\MediVision-AI\.venv\Lib\site-packages\sklearn\model_selection\_split.py:86: UserWarning: The groups parameter is ignored by KFold
  warnings.warn(
d:\depi_graduation_project\MediVision-AI\.venv\Lib\site-packages\sklearn\model_selection\_split.py:86: UserWarning: The groups parameter is ignored by KFold
  warnings.warn(


{'Logistic Regression': {'Mean F1 Macro': np.float64(0.34731315768370863),
  'Std F1 Macro': np.float64(0.004040964305163785)},
 'Random Forest': {'Mean F1 Macro': np.float64(0.287058250382593),
  'Std F1 Macro': np.float64(0.003139264163592272)},
 'XGBoost': {'Mean F1 Macro': np.float64(0.34475918707028363),
  'Std F1 Macro': np.float64(0.004023308681020193)}}

In [71]:
print(y_train[:10])

[[0 0 0]
 [0 0 0]
 [0 0 0]
 [0 0 0]
 [1 0 1]
 [1 0 0]
 [0 0 0]
 [0 0 1]
 [0 0 0]
 [0 0 1]]


In [72]:
print(y_train.sum(axis=0))

[5298 5252 9251]
